# Triaxial Permeation — Profile Evolution & Transport

Analysis for `triaxial_permeation.lmp` — a piston-driven, **constant-pressure**
flow of solvent through a laterally-periodic crosslinked slab.

There is **no strain sweep** (the whole production run is one continuous
permeation drive), so every field is shown as a **time evolution** over the
production frames (colour = timestep, cividis; bold black = final frame).

A **Phase 1.5 zero-flux reference** (equilibrated gel at `P_target`, piston
floating, before it repositions or loads) is recorded by the `.lmp` into `_ref`
files — mirroring the ε = 0 reference in `triaxial_compression.lmp`.  Its mean
profile (± CI) is overlaid as a dashed **baseline** on the stress and density
evolutions, and gel thickness is recorded from that pre-motion state so the
baseline `L₀` is visible before the piston moves.  All overlays degrade
gracefully: on older runs without `_ref` files they are simply skipped.

Sections:

1. Piston position & velocity vs. timestep
2. Gel z-thickness vs. timestep
3. Total stress (solvent + polymer) — σ_zz, σ_xx, σ_yy profile evolution
4. Solvent mass-density profile evolution
5. Cumulative solvent through the support + flux rate vs. timestep
6. Solvent **partial** stress vs. solvent-**only** (ss) stress — σ_zz, σ_xx, σ_yy and the isotropic average, each shown as an evolution

Network (effective) stress, pore pressure, permeability and cooperative
diffusivity are deferred to a later pass.

## Before running: files to copy from the cluster

Copy the run's output from the cluster into your local `flow_data_local` tree.
Set `RUN_ID`, `DATANAME`, `INTERACTION` (and optionally `NSTEPS`) in the
**Config** cell to match the run; the notebook builds every path from those.
Permeation writes **no** `_c<level>` tag and **no** `_ref` files, so filenames
are just `<name>_<DATANAME>_<INTERACTION>_<NSTEPS>`.

**Into** `flow_data_local/permeation/<RUN_ID>/`  *(cluster:* `.../output_files/`*)*

| file pattern | cluster subdir |
|---|---|
| `sigma{zz,xx,yy}_{polymer,solvent}_<sim>.dat` | `stress_data/` |
| `piston_position_<sim>.dat`, `piston_velocity_<sim>.dat` | `piston_data/` |
| `piston_force_<sim>.dat`, `piston_force_avg_<sim>.dat` | `piston_data/` |
| `permeate_count_<sim>.dat` | `permeation_data/` |
| `strain_zz_<sim>.dat` | `stress_data/` |
| `gel_dimensions_rg_<sim>.dat`, `box_dimensions_<sim>.dat` | `volume_data/` |
| `solvent_density_z_<DATANAME>_<INTERACTION>_<TOTSTEPS>.dat` | `chemical_potential/` |
| `pairs_<sim>.dump` | `pair_data/` |

**Into** `flow_data_local/traj_files.nosync/`  *(cluster scratch or* `traj_files/`*)*

| file pattern | note |
|---|---|
| `traj_stress_<sim>.lammpstrj` | position sync for the ss-stress pair dump |

The **Sync** cell below can pull all of these from Expanse automatically.

In [ ]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from scipy.interpolate import interp1d
from pathlib import Path

# ---- user rcParams (matches compression_analysis.ipynb) ----
plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False,
})

# ---- colorblind-friendly palettes ----
# Wong (2011) categorical palette for single-curve plots.
WONG = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','vermillion':'#D55E00',
        'skyblue':'#56B4E9','yellow':'#F0E442','reddishpurple':'#CC79A7','black':'#000000'}
# 'cividis' is the most CVD-safe sequential map -> used for the time gradient.
EVO_CMAP = 'cividis'
GEL_SHADE = dict(color='0.6', alpha=0.15, zorder=0)   # neutral grey, CVD-safe

print('Imports + style ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — only change the lines in this block to switch datasets
# ══════════════════════════════════════════════════════════════════════════
# sim_name is the exact suffix LAMMPS appends to every permeation output file:
#     <DATANAME>_<INTERACTION>_<NSTEPS>
# Permeation writes NO _c<level> tag (no sweep) and NO _ref files.
DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000000"
INTERACTION = "1.0_1.0"                 # epsSS_epsSP for the permeation run
NSTEPS      = 800000                      # permeation production steps; None -> auto-detect from files
RUN_ID      = "periodic_rho04_perm1"    # local folder label under flow_data_local/{permeation,plots}
# ══════════════════════════════════════════════════════════════════════════

DATA_DIR  = Path("../../flow_data_local/permeation") / RUN_ID
PLOT_DIR  = Path("../../flow_data_local/plots/permeation") / RUN_ID
TRAJ_DIR  = Path("../../flow_data_local/traj_files.nosync")
# auto-create the local folders for this run (idempotent; safe to re-run)
for _d in (DATA_DIR, PLOT_DIR, TRAJ_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print('folders ready:', DATA_DIR, '|', PLOT_DIR, '|', TRAJ_DIR)

# Build sim_name.  If NSTEPS is None, auto-detect from any component-stress file
# already present in DATA_DIR (e.g. after the Expanse sync); else use it directly.
if NSTEPS is None:
    import re as _re
    _cands = sorted(DATA_DIR.glob(f'sigmazz_solvent_{DATANAME}_{INTERACTION}_*.dat'))
    if not _cands:
        raise ValueError('NSTEPS is None and no sigmazz_solvent_*.dat in DATA_DIR yet '
                         '-- set NSTEPS explicitly, or run the Expanse sync cell first.')
    NSTEPS = int(_re.match(rf'.*_{_re.escape(INTERACTION)}_(\d+)\.dat$', _cands[-1].name).group(1))
    print(f'auto-detected NSTEPS = {NSTEPS} from {_cands[-1].name}')
sim_name = f'{DATANAME}_{INTERACTION}_{NSTEPS}'

# ---- analysis parameters ----
binWidth      = 2.0      # z-bin (sigma); must match triaxial_permeation.lmp
solvent_mass  = 1.0      # solvent bead mass (LJ); rho_mass = solvent_mass * rho_number
n_curves      = 10       # target time-evolution curves (matches num_stress_curves)
ci_level      = 0.95
gel_thresh    = 0.05     # gel interior = bins where |sigma_p,zz| > gel_thresh*max
flat_tol      = 0.15     # final evolution curve "flat inside gel" if rel. spread < this
wall_margin   = 4.0      # sigma trimmed off support/piston ends before flat test + mean

# ---- file paths (permeation: NO _c tag, NO _ref) ----
def D(name):     return DATA_DIR / f'{name}_{sim_name}.dat'
def Ddump(name): return DATA_DIR / f'{name}_{sim_name}.dump'
def T(name):     return TRAJ_DIR / f'{name}_{sim_name}.lammpstrj'

# component partial stresses (group-based, virial-only): sigma_{zz,xx,yy} poly/solv
F_SZZ_P = D('sigmazz_polymer');  F_SZZ_S = D('sigmazz_solvent')
F_SXX_P = D('sigmaxx_polymer');  F_SXX_S = D('sigmaxx_solvent')
F_SYY_P = D('sigmayy_polymer');  F_SYY_S = D('sigmayy_solvent')
# solvent-only (ss) stress: pair/local force dump + companion position dump
F_PAIRS      = Ddump('pairs')
F_TRAJSTRESS = T('traj_stress')
# solvent density lives in chemical_potential/ and is named with TOTSTEPS
# (not NSTEPS), so match by prefix + wildcard and take the newest.
def _find_density():
    cands = sorted(DATA_DIR.glob(f'solvent_density_z_{DATANAME}_{INTERACTION}_*.dat'))
    return cands[-1] if cands else DATA_DIR / f'solvent_density_z_{sim_name}.dat'
F_DENS = _find_density()
# piston, permeate, strain, gel/box dimensions
F_PISTON_POS       = D('piston_position')
F_PISTON_VEL       = D('piston_velocity')
F_PISTON_FORCE     = D('piston_force')
F_PISTON_FORCE_AVG = D('piston_force_avg')
F_PERMEATE         = D('permeate_count')
F_STRAIN           = D('strain_zz')
F_GELDIMS_RG       = D('gel_dimensions_rg')
F_BOXDIMS          = D('box_dimensions')

# ---- reference-state files (Phase 1.5 zero-flux baseline; may be absent on
#      old runs made before the reference block was added to the .lmp) ----
def Dref(name): return DATA_DIR / f'{name}_ref_{sim_name}.dat'
F_SZZ_P_REF = Dref('sigmazz_polymer'); F_SZZ_S_REF = Dref('sigmazz_solvent')
F_SXX_P_REF = Dref('sigmaxx_polymer'); F_SXX_S_REF = Dref('sigmaxx_solvent')
F_SYY_P_REF = Dref('sigmayy_polymer'); F_SYY_S_REF = Dref('sigmayy_solvent')
F_DENS_REF  = DATA_DIR / f'solvent_density_z_ref_{sim_name}.dat'
F_PAIRS_REF = DATA_DIR / f'pairs_ref_{sim_name}.dump'
F_TRAJREF   = TRAJ_DIR / f'traj_ref_{sim_name}.lammpstrj'
print('Config set for', sim_name)

## Sync data from Expanse (only when needed)

In [ ]:
# === Sync triaxial-permeation data from Expanse (only when needed) ===
# Same idiom as triaxial_compression.ipynb, adapted to permeation: no _c<level>
# tags, no _ref files, and the solvent-density file is matched by prefix (its
# name carries TOTSTEPS, not NSTEPS).  Files are staged on the cluster by
# basename glob (newest match per pattern), then pulled by SFTP.
#   • core .dat : sigma{zz,xx,yy}_{polymer,solvent}, piston_{position,velocity,
#                 force,force_avg}, permeate_count, strain_zz,
#                 gel_dimensions_rg, box_dimensions, solvent_density_z*  -> DATA_DIR
#   • pair dump : pairs (.dump)                                          -> DATA_DIR
#   • trajs     : traj_stress (.lammpstrj)                               -> TRAJ_DIR
# An Expanse login happens ONLY if a REQUIRED core file is missing locally.
# Set FORCE_SYNC = True to refresh the large pair/traj files too.
import paramiko, getpass, stat
from pathlib import Path

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
RUNS_ROOT    = "/home/dpollard/Documents/lammps_runs/triaxial_permeation"        # working dirs (output_files/*)
TRAJ_ROOT    = "/expanse/lustre/scratch/dpollard/temp_project/lammps_trajectories"  # .lammpstrj on scratch
STAGE_DIR    = f"{RUNS_ROOT}/permeation_stage"
FORCE_SYNC   = False     # True -> sync even if core files already present

# Basename GLOB patterns the loader reads, split by local destination.
_DATA_PATTERNS = [
    f'sigmazz_polymer_{sim_name}.dat', f'sigmazz_solvent_{sim_name}.dat',
    f'sigmaxx_polymer_{sim_name}.dat', f'sigmaxx_solvent_{sim_name}.dat',
    f'sigmayy_polymer_{sim_name}.dat', f'sigmayy_solvent_{sim_name}.dat',
    f'piston_position_{sim_name}.dat', f'piston_velocity_{sim_name}.dat',
    f'piston_force_{sim_name}.dat',    f'piston_force_avg_{sim_name}.dat',
    f'permeate_count_{sim_name}.dat',  f'strain_zz_{sim_name}.dat',
    f'gel_dimensions_rg_{sim_name}.dat', f'box_dimensions_{sim_name}.dat',
    f'solvent_density_z_{DATANAME}_{INTERACTION}_*.dat',   # TOTSTEPS wildcard
    f'pairs_{sim_name}.dump',
    # reference-state (Phase 1.5) files
    f'sigmazz_polymer_ref_{sim_name}.dat', f'sigmazz_solvent_ref_{sim_name}.dat',
    f'sigmaxx_polymer_ref_{sim_name}.dat', f'sigmaxx_solvent_ref_{sim_name}.dat',
    f'sigmayy_polymer_ref_{sim_name}.dat', f'sigmayy_solvent_ref_{sim_name}.dat',
    f'solvent_density_z_ref_{sim_name}.dat', f'pairs_ref_{sim_name}.dump',
]
_TRAJ_PATTERNS = [f'traj_stress_{sim_name}.lammpstrj', f'traj_ref_{sim_name}.lammpstrj']

# REQUIRED = the minimum needed for the main plots (glob-matched locally).
_REQUIRED_PATTERNS = [f'sigmazz_solvent_{sim_name}.dat',
                      f'piston_position_{sim_name}.dat',
                      f'permeate_count_{sim_name}.dat']
def _have(pat, d):  return bool(sorted(Path(d).glob(pat)))
missing_req = [p for p in _REQUIRED_PATTERNS if not _have(p, DATA_DIR)]
missing_all = ([p for p in _DATA_PATTERNS if not _have(p, DATA_DIR)]
             + [p for p in _TRAJ_PATTERNS if not _have(p, TRAJ_DIR)])

if not FORCE_SYNC and not missing_req:
    print(f"All required files present locally "
          f"({len(missing_all)} optional pattern(s) unmatched) — skipping Expanse login.")
else:
    why = "FORCE_SYNC" if (FORCE_SYNC and not missing_req) else f"{len(missing_req)} required pattern(s) missing"
    print(f"Syncing from Expanse ({why}); {len(missing_all)} of "
          f"{len(_DATA_PATTERNS)+len(_TRAJ_PATTERNS)} target patterns unmatched locally.")

    DATA_PATS = " ".join(f'"{p}"' for p in _DATA_PATTERNS)
    TRAJ_PATS = " ".join(f'"{p}"' for p in _TRAJ_PATTERNS)

    # Stage the newest match for each basename glob with ONE find per tree.
    # cp -p preserves mtimes so the SFTP skip fires on reruns.
    stage_script = r'''
set -u
RUNS="__RUNS__"; TRAJ="__TRAJ__"; STAGE="__STAGE__"
rm -rf "$STAGE"; mkdir -p "$STAGE/data" "$STAGE/traj"
for PAT in __DATA_PATS__; do
  S=$(find "$RUNS" -name "$PAT" -not -path '*/permeation_stage/*' -printf '%T@ %p\n' 2>/dev/null | sort -rn | head -1 | cut -d' ' -f2-)
  [ -n "$S" ] && cp -p "$S" "$STAGE/data/" 2>/dev/null || true
done
for PAT in __TRAJ_PATS__; do
  S=$(find "$TRAJ" "$RUNS" -name "$PAT" -not -path '*/permeation_stage/*' -printf '%T@ %p\n' 2>/dev/null | sort -rn | head -1 | cut -d' ' -f2-)
  [ -n "$S" ] && cp -p "$S" "$STAGE/traj/" 2>/dev/null || true
done
echo "  staged: $(ls "$STAGE/data" 2>/dev/null | wc -l) data, $(ls "$STAGE/traj" 2>/dev/null | wc -l) traj"
'''
    stage_script = (stage_script.replace("__RUNS__", RUNS_ROOT).replace("__TRAJ__", TRAJ_ROOT)
                    .replace("__STAGE__", STAGE_DIR).replace("__DATA_PATS__", DATA_PATS)
                    .replace("__TRAJ_PATS__", TRAJ_PATS))

    password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
    totp     = getpass.getpass("TOTP / verification code: ")
    def auth_handler(title, instructions, prompt_list):
        return [password if "password" in p.strip().lower() else totp for p, _ in prompt_list]

    print("Connecting to Expanse...")
    transport = paramiko.Transport((EXPANSE_HOST, 22))
    transport.connect()
    transport.auth_interactive(EXPANSE_USER, auth_handler)
    ssh = paramiko.SSHClient(); ssh._transport = transport

    print("Step 1 — staging files on Expanse...")
    _, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
    stdout.channel.sendall(stage_script.encode()); stdout.channel.shutdown_write()
    print(stdout.read().decode())

    print("Step 2 — downloading via SFTP (skips files already present)...")
    sftp = ssh.open_sftp()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    TRAJ_DIR.mkdir(parents=True, exist_ok=True)
    def sftp_pull(remote_dir, local_dir):
        local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)
        try: entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError: return
        for e in entries:
            rp, lp = f"{remote_dir}/{e.filename}", local_dir / e.filename
            if stat.S_ISDIR(e.st_mode):
                sftp_pull(rp, lp); continue
            if lp.exists() and lp.stat().st_mtime >= e.st_mtime:
                continue
            sftp.get(rp, str(lp))
    sftp_pull(f"{STAGE_DIR}/data", DATA_DIR)
    sftp_pull(f"{STAGE_DIR}/traj", TRAJ_DIR)
    sftp.close(); ssh.close()
    print("Sync complete.")
    # refresh the density path now that files may have landed
    F_DENS = _find_density()

In [ ]:
# ============================ HELPER FUNCTIONS =============================
def read_print_file(filepath, col_names=None):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    if col_names is None: col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}

def read_ave_time_file(filepath):
    # fix ave/time mode vector -> list of (timestep, bin_idx, values).
    out = []
    with open(filepath) as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            ts, nrows = int(parts[0]), int(parts[1])
            vals = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2: vals.append(float(vp[1]))
            if vals: out.append((ts, np.arange(1, len(vals)+1), np.array(vals)))
            i += nrows + 1
        else:
            i += 1
    return out

def read_ave_chunk_file(filepath):
    # fix ave/chunk -> list of (timestep, array[rows, cols]).
    # cols: [chunk_id, Coord1, Ncount, val1(, val2...)].
    snaps = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) in (2, 3):
            try: ts, nch = int(parts[0]), int(parts[1])
            except ValueError:
                i += 1; continue
            rows = []
            for j in range(1, nch + 1):
                if i + j < len(lines): rows.append([float(v) for v in lines[i + j].split()])
            if rows: snaps.append((ts, np.array(rows)))
            i += nch + 1
        else:
            i += 1
    return snaps

def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    # cols: step  L_initial  L_current  -> eps = (L0 - L)/L0
    ts = arr[:, 0].astype(int); L0 = arr[:, 1]; L = arr[:, 2]
    eps = (L0 - L) / L0
    return ts, L0, L, eps

def mean_ci(stack, ci=0.95):
    # stack: (n_samples, n_bins) -> (mean, lo, hi) per bin via t-interval.
    stack = np.asarray(stack, float)
    n = stack.shape[0]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        m = np.nanmean(stack, axis=0)
        if n < 2:
            return m, m, m
        se = stats.sem(stack, axis=0, nan_policy='omit')
    half = se * stats.t.ppf(0.5 + ci/2, df=n-1)
    return m, m - half, m + half

def rolling_mean(y, win):
    # Centered moving average; window shrinks at the edges (min_periods=1).
    y = np.asarray(y, float); n = len(y)
    if win <= 1 or n == 0:
        return y.copy()
    half = win // 2
    csum = np.concatenate(([0.0], np.cumsum(y)))
    out  = np.empty(n)
    for k in range(n):
        lo, hi = max(0, k - half), min(n, k + half + 1)
        out[k] = (csum[hi] - csum[lo]) / (hi - lo)
    return out

def read_pairs_local_dump(filepath):
    # dump local (pair/local) -> list (timestep, box, data[n,7])
    # data cols: id1 id2 type1 type2 fx fy fz.
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 7))))
            i = s + n
        else:
            i += 1
    return frames

def _stream_traj_positions(traj_file, target_ts, types_keep):
    # Return {ts: (box, {id:(x,y,z)})} for atoms whose type is in types_keep.
    out = {}
    with open(traj_file) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if 'ITEM: TIMESTEP' in lines[i]:
            ts = int(lines[i+1]); n = int(lines[i+3])
            if ts in target_ts:
                xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
                zb = list(map(float, lines[i+7].split()))
                box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
                pos = {}
                for j in range(n):
                    p = lines[i+9+j].split()
                    if int(p[1]) in types_keep:
                        pos[int(p[0])] = (float(p[3]), float(p[4]), float(p[5]))
                out[ts] = (box, pos)
            i += 9 + n
        else:
            i += 1
    return out

print('Readers defined')

In [ ]:
# ===== solvent-only (ss) stress components from the pair/local force dump =====
# compute stress/atom cannot sub-style-filter in this LAMMPS build, so the
# solvent-solvent virial is reconstructed here from per-pair forces + positions
# (same method as compression_analysis.ipynb, generalised to xx/yy/zz).  No
# kinetic term (virial only); no bond term (solvent has no bonds).
def _virial_components_from_pairs(pair_data, pos, box, binWidth, type_filter):
    # sigma_xx,yy,zz(z) from pair/local forces, both atoms in type_filter (a set).
    # Returns (z_bins, sxx, syy, szz) -- virial only.
    # Sign convention: sigma_aa = -(dr_a * f_a)/V  (positive => compressive),
    # matching the group-based partial stresses sigma_s_aa in triaxial_permeation.lmp.
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    z_bins = zlo + (np.arange(nb)+0.5)*binWidth
    sxx = np.zeros(nb); syy = np.zeros(nb); szz = np.zeros(nb)
    if len(pair_data):
        m = np.isin(pair_data[:,2].astype(int), list(type_filter)) & \
            np.isin(pair_data[:,3].astype(int), list(type_filter))
        sp = pair_data[m]
        if len(sp):
            id1 = sp[:,0].astype(int); id2 = sp[:,1].astype(int)
            fx = sp[:,4]; fy = sp[:,5]; fz = sp[:,6]
            keep = np.array([(a in pos and b in pos) for a,b in zip(id1,id2)])
            if keep.any():
                id1=id1[keep]; id2=id2[keep]; fx=fx[keep]; fy=fy[keep]; fz=fz[keep]
                p1 = np.array([pos[a] for a in id1]); p2 = np.array([pos[b] for b in id2])
                dx = p2[:,0]-p1[:,0]; dx -= lx*np.round(dx/lx)
                dy = p2[:,1]-p1[:,1]; dy -= ly*np.round(dy/ly)
                dz = p2[:,2]-p1[:,2]; dz -= lz*np.round(dz/lz)
                wxx = -(dx*fx); wyy = -(dy*fy); wzz = -(dz*fz)
                zmid = p1[:,2] + dz*0.5
                bi = ((zmid - zlo)/binWidth).astype(int)
                ok = (bi>=0)&(bi<nb)
                np.add.at(sxx, bi[ok], wxx[ok])
                np.add.at(syy, bi[ok], wyy[ok])
                np.add.at(szz, bi[ok], wzz[ok])
    return z_bins, sxx/binvol, syy/binvol, szz/binvol

def compute_ss_components(pairs_file, traj_file, binWidth, z_target):
    # Per-frame sigma_s,ss xx/yy/zz(z), virial only, interpolated onto z_target.
    # Returns (timesteps, stack_xx, stack_yy, stack_zz), each stack (n_frames, nz).
    pf = read_pairs_local_dump(pairs_file)
    if not pf:
        z0 = np.zeros((0, len(z_target)))
        return np.array([]), z0, z0.copy(), z0.copy()
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {3})
    ts_out, sx, sy, sz = [], [], [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, vxx, vyy, vzz = _virial_components_from_pairs(pdata, pos, box, binWidth, {3})
        sx.append(np.interp(z_target, zb, vxx, left=np.nan, right=np.nan))
        sy.append(np.interp(z_target, zb, vyy, left=np.nan, right=np.nan))
        sz.append(np.interp(z_target, zb, vzz, left=np.nan, right=np.nan))
        ts_out.append(ts)
    return np.array(ts_out), np.array(sx), np.array(sy), np.array(sz)

print('ss stress-component reconstruction defined')

## Load data: box/wall geometry, component stresses, density, ss stress

In [ ]:
# ---- box z-extent (fixed: only the piston moves) + wall positions ----
def _read_box_z(dumpfile):
    # Return (zlo, zhi) from the first frame of a LAMMPS dump / lammpstrj.
    with open(dumpfile) as f:
        head = [next(f) for _ in range(9)]
    return tuple(map(float, head[7].split()))          # line 7 = z BOX BOUNDS

def _read_box_xy(dumpfile):
    # Return (lx, ly) from the first frame of a LAMMPS dump / lammpstrj header.
    with open(dumpfile) as f:
        head = [next(f) for _ in range(9)]
    xlo, xhi = map(float, head[5].split())   # line 5 = x BOX BOUNDS
    ylo, yhi = map(float, head[6].split())   # line 6 = y BOX BOUNDS
    return (xhi - xlo), (yhi - ylo)

# geometry source: prefer the pair dump, fall back to the stress-sync traj
_geom_src = F_PAIRS if F_PAIRS.exists() else F_TRAJSTRESS
Z_LO, Z_HI = _read_box_z(_geom_src)
LX, LY     = _read_box_xy(_geom_src)
LZ = Z_HI - Z_LO
piston_area = LX * LY
zn = lambda z: (np.asarray(z, float) - Z_LO) / LZ      # fractional box height, ~[0, 1]
print(f'box: z in [{Z_LO:.2f}, {Z_HI:.2f}]  Lz={LZ:.2f} | A = lx*ly = {LX:.2f} x {LY:.2f} = {piston_area:.2f}')

def _wall_z_first_frame(traj, types=(4, 5)):
    # Mean z of each atom type in the FIRST frame of a LAMMPS dump trajectory.
    zc = {t: [] for t in types}
    if not Path(traj).exists():
        return {t: np.nan for t in types}
    with open(traj) as f:
        cols = None
        for line in f:                                  # advance to the ATOMS header
            if line.startswith('ITEM: ATOMS'):
                cols = line.split()[2:]; break
        ti, zi = cols.index('type'), cols.index('z')
        for line in f:
            if line.startswith('ITEM:'): break          # stop at next frame
            p = line.split(); t = int(float(p[ti]))
            if t in zc: zc[t].append(float(p[zi]))
    return {t: (np.mean(v) if v else np.nan) for t, v in zc.items()}
_wz = _wall_z_first_frame(F_TRAJSTRESS, (4, 5))
z_support = _wz.get(4, np.nan)                          # type 4 = frozen support sheet
z_piston  = _wz.get(5, np.nan)                          # type 5 = piston sheet (starting z)
print(f'support (type4) z = {z_support:.2f}  |  piston (type5) start z = {z_piston:.2f}')

# piston z(t): recorded every volume_freq through production; interp clamps to the
# endpoints, so any stress-frame timestep maps to the piston's true position.
if Path(F_PISTON_POS).exists():
    _pp = np.loadtxt(F_PISTON_POS, comments='#')
    _pt, _pz = _pp[:, 0], _pp[:, 1]
    def piston_z_at(t): return float(np.interp(t, _pt, _pz))
    print(f'piston z(t): {_pt[0]:.0f} -> {_pt[-1]:.0f} steps,  z {_pz[0]:.1f} -> {_pz[-1]:.1f}')
else:
    def piston_z_at(t): return z_piston
    print('piston z(t): file missing (using start position)')

# ---- component partial stresses (production): z-grid + total/partial series ----
# Each file is a fix ave/time mode-vector (reduce/chunk) => per-bin sigma_aa(z,t).
def _load_comp(path):
    return read_ave_time_file(path)

szz_p = _load_comp(F_SZZ_P); szz_s = _load_comp(F_SZZ_S)
sxx_p = _load_comp(F_SXX_P); sxx_s = _load_comp(F_SXX_S)
syy_p = _load_comp(F_SYY_P); syy_s = _load_comp(F_SYY_S)

n_prod   = len(szz_p)
prod_ts  = np.array([szz_p[i][0] for i in range(n_prod)])
bins_z   = szz_p[0][1]
z_coords = Z_LO + bins_z * binWidth - binWidth/2.0     # bin centers in real box coords

# partial (group-based) stresses per component, per frame
sig_p_zz = [szz_p[i][2] for i in range(n_prod)]
sig_s_zz = [szz_s[i][2] for i in range(n_prod)]
sig_p_xx = [sxx_p[i][2] for i in range(n_prod)]
sig_s_xx = [sxx_s[i][2] for i in range(n_prod)]
sig_p_yy = [syy_p[i][2] for i in range(n_prod)]
sig_s_yy = [syy_s[i][2] for i in range(n_prod)]
# TOTAL stress = polymer + solvent, per component
sig_t_zz = [sig_p_zz[i] + sig_s_zz[i] for i in range(n_prod)]
sig_t_xx = [sig_p_xx[i] + sig_s_xx[i] for i in range(n_prod)]
sig_t_yy = [sig_p_yy[i] + sig_s_yy[i] for i in range(n_prod)]
print(f'component stress: {n_prod} production snapshots, {len(z_coords)} z-bins '
      f'[{z_coords.min():.1f}, {z_coords.max():.1f}]')

# ---- gel interior from the time-averaged polymer sigma_zz ----
# (No eps=0 reference in permeation, so the gel extent is read from the mean
# polymer stress over the production run.)
sig_p_zz_mean = np.nanmean(np.array(sig_p_zz), axis=0)
_pm = np.abs(sig_p_zz_mean); _pmax = float(np.nanmax(_pm))
_gel = (_pm > gel_thresh*_pmax) if _pmax > 0 else np.zeros(len(z_coords), bool)
z_gel_lo = float(z_coords[_gel].min()) if _gel.any() else z_coords[0]
z_gel_hi = float(z_coords[_gel].max()) if _gel.any() else z_coords[-1]
in_gel   = (z_coords >= z_gel_lo) & (z_coords <= z_gel_hi)
print(f'gel interior (mean polymer stress): z in [{z_gel_lo:.1f}, {z_gel_hi:.1f}]  ({in_gel.sum()} bins)')

In [ ]:
# ---- solvent density (production): cols chunk, Coord1, Ncount, density/number ----
# triaxial_permeation.lmp writes density/number only; solvent bead mass = 1, so
# the mass density equals the number density up to the (unit) solvent_mass factor.
def _load_density_number(path):
    snaps = read_ave_chunk_file(path)
    ts = np.array([s[0] for s in snaps])
    z  = snaps[0][1][:, 1]
    nden = np.array([s[1][:, 3] for s in snaps])        # density/number
    return ts, z, nden

dens_ts, dens_z, dens_n = _load_density_number(F_DENS)
dens_m = dens_n * solvent_mass                          # mass density (= number density, m=1)
print(f'density: {len(dens_ts)} production snapshots on {len(dens_z)} bins '
      f'(t={dens_ts[0]} -> {dens_ts[-1]})')

# bulk reference solvent mass density rho_{s,0} from the first frame's reservoir
# bins (used only for reference lines / normalisation if wanted later).
_rmax = float(np.nanmax(dens_m[0])); _res = dens_m[0] >= 0.85*_rmax
rho_s0 = float(np.nanmean(dens_m[0][_res])) if _res.any() else float(np.nanmax(dens_m[0]))
print(f'rho_s,0 (first-frame bulk reservoir mass density) = {rho_s0:.4f}')

In [ ]:
# ---- solvent-only (ss) stress components: production ----
# Reconstructed from the pair/local force dump + companion position traj.
ss_ts = None; ss_xx = ss_yy = ss_zz = ss_iso = None
if F_PAIRS.exists() and F_TRAJSTRESS.exists():
    ss_ts, ss_xx, ss_yy, ss_zz = compute_ss_components(F_PAIRS, F_TRAJSTRESS, binWidth, z_coords)
    if len(ss_ts):
        ss_iso = (ss_xx + ss_yy + ss_zz) / 3.0
        print(f'sigma_s,ss production: {len(ss_ts)} frames')
    else:
        print('NOTE: ss dumps present but no overlapping frames found')
        ss_ts = None
else:
    print('NOTE: ss dumps missing -', F_PAIRS.name, '/', F_TRAJSTRESS.name,
          '\n      -> Section 6 right column (solvent-only) will be skipped.')

In [ ]:
# ---- reference-state (Phase 1.5 zero-flux baseline): mean + CI per profile ----
# All optional: absent on runs made before the reference block was added to the
# .lmp, in which case every overlay below is silently skipped (ref_* = None).
def _ref_stack(path):
    if not Path(path).exists():
        return None
    recs = read_ave_time_file(path)
    return np.array([r[2] for r in recs]) if recs else None

_rp = {'zz': _ref_stack(F_SZZ_P_REF), 'xx': _ref_stack(F_SXX_P_REF), 'yy': _ref_stack(F_SYY_P_REF)}
_rs = {'zz': _ref_stack(F_SZZ_S_REF), 'xx': _ref_stack(F_SXX_S_REF), 'yy': _ref_stack(F_SYY_S_REF)}
has_ref_stress = all(v is not None and len(v) for v in list(_rp.values()) + list(_rs.values()))
if has_ref_stress:
    ref_tot_ci     = {c: mean_ci(_rp[c] + _rs[c], ci_level) for c in ('zz', 'xx', 'yy')}   # total
    ref_solv_ci    = {c: mean_ci(_rs[c],          ci_level) for c in ('zz', 'xx', 'yy')}   # solvent partial
    ref_solv_iso_ci = mean_ci((_rs['xx'] + _rs['yy'] + _rs['zz']) / 3.0, ci_level)
    print(f'reference stress: loaded ({_rp["zz"].shape[0]} snapshots)')
else:
    ref_tot_ci = ref_solv_ci = None; ref_solv_iso_ci = None
    print('reference stress: not available (run the updated .lmp with the Phase 1.5 block)')

# reference solvent density (mean + CI)
if Path(F_DENS_REF).exists():
    _rt, ref_dens_z, _rn = _load_density_number(F_DENS_REF)
    ref_dens_ci = mean_ci(_rn * solvent_mass, ci_level)
    print(f'reference density: loaded ({_rn.shape[0]} snapshots)')
else:
    ref_dens_z = None; ref_dens_ci = None
    print('reference density: not available')

# reference solvent-only (ss) stress (mean + CI per component + isotropic)
ref_ss_ci = None; ref_ss_iso_ci = None
if F_PAIRS_REF.exists() and F_TRAJREF.exists():
    _t, _xx, _yy, _zz = compute_ss_components(F_PAIRS_REF, F_TRAJREF, binWidth, z_coords)
    if len(_t):
        ref_ss_ci = {'zz': mean_ci(_zz, ci_level), 'xx': mean_ci(_xx, ci_level), 'yy': mean_ci(_yy, ci_level)}
        ref_ss_iso_ci = mean_ci((_xx + _yy + _zz) / 3.0, ci_level)
        print(f'reference ss stress: loaded ({len(_t)} frames)')
    else:
        print('reference ss stress: dumps present but no overlapping frames')
else:
    print('reference ss stress: not available')

## Plotting helpers

In [ ]:
def _fmt_val_unc(v, u):
    # value +/- uncertainty, uncertainty rounded to 2 sig figs (value matched).
    v = float(v)
    if np.isfinite(u) and u > 0:
        dec = int(np.clip(1 - np.floor(np.log10(u)), 0, 6))
        return f'{v:.{dec}f} ± {u:.{dec}f}'
    return f'{v:.3g}'

def _fmt_mu(vals):
    # mean +/- std (2 sig figs) over finite values.
    a = np.asarray(vals, float); a = a[np.isfinite(a)]
    if a.size == 0: return 'n/a'
    return _fmt_val_unc(np.mean(a), np.std(a))

def _final_flat_inside(curve, z, in_gel, tol):
    # Flat = small linear TREND across the gel (noise-robust).
    v = curve[in_gel]; zz = np.asarray(z)[in_gel]
    m = np.isfinite(v); v, zz = v[m], zz[m]
    if len(v) < 3: return False
    slope = np.polyfit(zz, v, 1)[0]
    rise = abs(slope) * (zz.max() - zz.min())
    denom = max(abs(np.mean(v)), 1e-9)
    return (rise / denom) < tol

def shade_gel(ax):
    ax.axvspan(zn(z_gel_lo), zn(z_gel_hi), **GEL_SHADE)

def mark_walls(ax, ts=None):
    # Frozen support (type4, solid) + piston (type5, dash-dot).  ts=None ->
    # starting piston position; ts given -> piston per evolution timestep, final
    # one dark and earlier ones faded.
    if np.isfinite(z_support):
        ax.axvline(zn(z_support), color=WONG['black'], ls='-', lw=1.5, alpha=0.85, zorder=4)
    if ts is None:
        pistons = [(z_piston, True)]
    else:
        ts = np.asarray(ts)
        pistons = [(piston_z_at(t), i == len(ts) - 1) for i, t in enumerate(ts)]
    for pz, is_last in pistons:
        if np.isfinite(pz):
            ax.axvline(zn(pz), color=('0.15' if is_last else '0.7'), ls='-.',
                       lw=(1.6 if is_last else 1.0), alpha=(0.9 if is_last else 0.35),
                       zorder=(4 if is_last else 3))

def subsample(ts, stack, k):
    # Evenly pick up to k snapshots (keep order; always include the last).
    ts = np.asarray(ts); stack = np.asarray(stack)
    n = len(ts)
    if n <= k: idx = np.arange(n)
    else:      idx = np.unique(np.linspace(0, n-1, k).round().astype(int))
    return ts[idx], stack[idx]

def plot_evolution(ax, z, ts, stack, ylabel, title, add_cbar=True):
    # Time-coloured profiles (cividis); final curve bold black; gel shaded.
    # Mean-in-gel annotation ONLY if the final curve is flat inside the gel.
    ts = np.asarray(ts); stack = np.asarray(stack)
    norm = Normalize(vmin=ts.min(), vmax=ts.max())
    cmap = plt.get_cmap(EVO_CMAP)
    zx = zn(z)
    for i in range(len(ts)):
        is_last = (i == len(ts)-1)
        ax.plot(zx, stack[i], '-',
                color=('k' if is_last else cmap(norm(ts[i]))),
                lw=(3.5 if is_last else 1.6),
                alpha=(1.0 if is_last else 0.75),
                zorder=(5 if is_last else 3))
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax); mark_walls(ax, ts)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    if add_cbar:
        sm = plt.cm.ScalarMappable(cmap=EVO_CMAP, norm=norm); sm.set_array([])
        cb = ax.figure.colorbar(sm, ax=ax, fraction=0.046, pad=0.02); cb.set_label('timestep')
    # gel is squeezed between the support and the CURRENT piston: restrict the
    # interior to that window so the reservoir past the piston never enters the
    # flatness test or the mean.
    z_pist = piston_z_at(np.asarray(ts)[-1])
    _zz = np.asarray(z)
    interior = in_gel & (_zz >= z_gel_lo + wall_margin) & (_zz <= z_pist - wall_margin)
    if interior.any() and _final_flat_inside(stack[-1], z, interior, flat_tol):
        ax.text(0.02, 0.03, 'final\nmean in gel = ' + _fmt_mu(stack[-1][interior]),
                transform=ax.transAxes, va='bottom', ha='left', fontsize=14,
                bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

def overlay_reference(ax, z, mlohi, color=WONG['black'], label='reference (baseline)'):
    # Overlay the Phase 1.5 zero-flux reference profile (dashed mean + light CI
    # band) on an evolution panel.  mlohi = (mean, lo, hi) or None (skips).
    if mlohi is None:
        return
    m, lo, hi = mlohi
    zx = zn(z)
    ax.fill_between(zx, lo, hi, color=color, alpha=0.15, lw=0, zorder=1)
    ax.plot(zx, m, '--', color=color, lw=2.2, alpha=0.9, zorder=4, label=label)
    ax.legend(fontsize=12, loc='upper right')

def robust_ylim(ax, curves, zmask=None, pad=0.12, qlo=2, qhi=98, include_zero=True):
    # Frame the y-axis to the bulk of the data (qlo-qhi percentiles), ignoring
    # extreme wall-edge spike bins.
    vals = []
    for c in curves:
        c = np.asarray(c, float)
        if zmask is not None: c = c[zmask]
        c = c[np.isfinite(c)]
        if c.size: vals.append(c)
    if not vals:
        return
    v = np.concatenate(vals)
    lo, hi = np.percentile(v, [qlo, qhi])
    if include_zero:
        lo, hi = min(lo, 0.0), max(hi, 0.0)
    if hi <= lo: hi = lo + 1.0
    d = (hi - lo) * pad
    ax.set_ylim(lo - d, hi + d)

print('Plot helpers ready')

## 1 — Piston position & velocity vs. timestep

The force-controlled piston descends as solvent permeates through the gel and
support.  `piston_position` is the piston COM z; `piston_velocity` is its mean
z-velocity (noisy under force control, so a rolling mean is overlaid).

In [ ]:
# ==========================================================================
#  1 — PISTON POSITION & VELOCITY  (from piston_position / piston_velocity)
# ==========================================================================
if Path(F_PISTON_POS).exists() and Path(F_PISTON_VEL).exists():
    _pp = read_print_file(F_PISTON_POS, col_names=['step', 'z'])
    _pv = read_print_file(F_PISTON_VEL, col_names=['step', 'vz'])
    step_pos, z_pos  = _pp['step'].astype(int), _pp['z']
    step_vel, vz     = _pv['step'].astype(int), _pv['vz']
    vz_roll = rolling_mean(vz, 21)

    fig, (axP, axV) = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    fig.suptitle(f'Piston position & velocity:  {sim_name}', fontsize=14, fontweight='bold')

    axP.plot(step_pos, z_pos, '-', color=WONG['blue'], lw=2.4)
    if np.isfinite(z_support):
        axP.axhline(z_support, color=WONG['black'], ls='-', lw=1.4, alpha=0.7, label='support')
        axP.legend(fontsize=13, loc='best')
    axP.set_xlabel('step'); axP.set_ylabel(r'piston $z$  ($\sigma$)')
    axP.set_title('(a) piston position'); axP.grid(alpha=0.3)

    axV.plot(step_vel, vz, '-', color=WONG['vermillion'], lw=1.0, alpha=0.30, label='raw')
    axV.plot(step_vel, vz_roll, '-', color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label='rolling mean (21 pts)')
    axV.axhline(0, color='k', ls='--', lw=0.9, alpha=0.5)
    axV.set_xlabel('step'); axV.set_ylabel(r'piston $v_z$  ($\sigma/\tau$)')
    axV.set_title('(b) piston velocity'); axV.grid(alpha=0.3); axV.legend(fontsize=13, loc='best')

    out = PLOT_DIR / f'piston_position_velocity_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
    print(f'  piston z: {z_pos[0]:.2f} -> {z_pos[-1]:.2f}  (dropped {z_pos[0]-z_pos[-1]:.2f} sigma)')
else:
    print('skipped — piston_position / piston_velocity file missing')

## 2 — Gel thickness (z) vs. timestep

Rg-based gel thickness $L_z = 2\sqrt{3}\,\sqrt{\langle R_{g,zz}^2\rangle}$ from
`gel_dimensions_rg` (robust to stray beads).  The `strain_zz` file's current
length is overlaid as a cross-check when present.

In [ ]:
# ==========================================================================
#  2 — GEL Z-THICKNESS  (Rg-based, from gel_dimensions_rg)
# ==========================================================================
if Path(F_GELDIMS_RG).exists():
    _gd = np.loadtxt(F_GELDIMS_RG, comments='#'); _gd = np.atleast_2d(_gd)
    g_step = _gd[:, 0].astype(int); g_lz = _gd[:, 3]   # cols: step lx ly lz

    fig, ax = plt.subplots(figsize=(9.5, 6.5), constrained_layout=True)
    fig.suptitle(f'Gel z-thickness:  {sim_name}', fontsize=14, fontweight='bold')
    ax.plot(g_step, g_lz, '-', color=WONG['green'], lw=2.6, label=r'$L_z$ (Rg)')

    # pre-drive baseline L0 (the .lmp now records from the equilibrated,
    # pre-motion state, so the series opens flat at L0 before the piston moves)
    L0 = float(g_lz[0])
    ax.axhline(L0, color=WONG['black'], ls=':', lw=1.6, alpha=0.7,
               label=rf'baseline $L_0={L0:.2f}\,\sigma$')

    # cross-check against strain_zz current length if available
    if Path(F_STRAIN).exists():
        s_ts, s_L0, s_L, s_eps = read_strain_file(F_STRAIN)
        ax.plot(s_ts, s_L, '--', color=WONG['black'], lw=1.6, alpha=0.7,
                label=r'$L_z$ (strain file)')
    ax.set_xlabel('step'); ax.set_ylabel(r'gel thickness $L_z$  ($\sigma$)')
    ax.grid(alpha=0.3); ax.legend(fontsize=13, loc='best')

    out = PLOT_DIR / f'gel_thickness_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
    print(f'  L_z: {g_lz[0]:.2f} -> {g_lz[-1]:.2f}  '
          f'(strain = {100*(g_lz[0]-g_lz[-1])/g_lz[0]:.2f} %)')
else:
    print('skipped — gel_dimensions_rg file missing')

## 3 — Total stress (solvent + polymer) profile evolution

Total normal stress $\sigma_{\alpha\alpha}^{t}(z,t)=\sigma_{p,\alpha\alpha}+\sigma_{s,\alpha\alpha}$
(group-based partial stresses summed) for $\alpha=z,x,y$.  Colour = timestep;
bold black = final frame; gel interior shaded; support/piston marked.

In [ ]:
# ==========================================================================
#  3 — TOTAL STRESS PROFILE EVOLUTION  (sigma_zz, sigma_xx, sigma_yy)
# ==========================================================================
fig, axes = plt.subplots(1, 3, figsize=(25, 7.5), constrained_layout=True)
fig.suptitle(f'Total stress evolution (solvent + polymer):  {sim_name}',
             fontsize=14, fontweight='bold')

for ax, tag, comp, label, ttl in [
    (axes[0], 'zz', sig_t_zz, r'$\sigma_{zz}^{t}(z,t)$', r'(a) $\sigma_{zz}^{t}=\sigma_{p,zz}+\sigma_{s,zz}$'),
    (axes[1], 'xx', sig_t_xx, r'$\sigma_{xx}^{t}(z,t)$', r'(b) $\sigma_{xx}^{t}=\sigma_{p,xx}+\sigma_{s,xx}$'),
    (axes[2], 'yy', sig_t_yy, r'$\sigma_{yy}^{t}(z,t)$', r'(c) $\sigma_{yy}^{t}=\sigma_{p,yy}+\sigma_{s,yy}$'),
]:
    ts_e, ev = subsample(prod_ts, np.array(comp), n_curves)
    plot_evolution(ax, z_coords, ts_e, ev, label, ttl)
    if ref_tot_ci is not None:
        overlay_reference(ax, z_coords, ref_tot_ci[tag])

out = PLOT_DIR / f'total_stress_evolution_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

## 4 — Solvent mass-density profile evolution

$\rho_s(z,t)$ from the `solvent_density_z` chunk average (density/number × bead
mass; the two are equal in these LJ units).  The feed reservoir thins as solvent
is driven through the gel and support.

In [ ]:
# ==========================================================================
#  4 — SOLVENT MASS-DENSITY PROFILE EVOLUTION
# ==========================================================================
fig, ax = plt.subplots(figsize=(9.5, 7), constrained_layout=True)
fig.suptitle(f'Solvent mass-density evolution:  {sim_name}', fontsize=14, fontweight='bold')

rho_ts, rho_ev = subsample(dens_ts, dens_m, n_curves)
plot_evolution(ax, dens_z, rho_ts, rho_ev,
               r'$\rho_s(z,t)\ (m\,\sigma^{-3})$', r'Solvent mass density $\rho_s$')
if ref_dens_ci is not None:
    overlay_reference(ax, ref_dens_z, ref_dens_ci)
ax.axhline(rho_s0, color=WONG['black'], ls=':', lw=1.5, alpha=0.7)
ax.text(0.98, 0.05, r'$\rho_{s,0}$'+f' = {rho_s0:.3f}', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=15)

out = PLOT_DIR / f'solvent_density_evolution_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

## 5 — Solvent through the support: cumulative count & flux rate

`permeate_count` records $N_{\rm perm}(t)$ = solvent that has crossed below the
support membrane (permeated out).  The **rate** $dN/dt$ is a centered finite
difference; the flux $J = (dN/dt)/(L_xL_y)$ normalises by the membrane area.

In [ ]:
# ==========================================================================
#  5 — PERMEATE COUNT (cumulative) + FLUX RATE
# ==========================================================================
if Path(F_PERMEATE).exists():
    _pc = read_print_file(F_PERMEATE, col_names=['step', 'N'])
    pstep = _pc['step'].astype(int); Nperm = _pc['N']

    # centered finite-difference rate dN/dt (per step); flux = rate / area
    if len(pstep) > 2:
        dN = np.gradient(Nperm.astype(float), pstep.astype(float))
    else:
        dN = np.zeros_like(Nperm, dtype=float)
    dN_roll = rolling_mean(dN, 21)
    flux = dN_roll / piston_area

    fig, (axC, axR) = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    fig.suptitle(f'Permeate through the support:  {sim_name}', fontsize=14, fontweight='bold')

    axC.plot(pstep, Nperm, '-', color=WONG['blue'], lw=2.6)
    axC.set_xlabel('step'); axC.set_ylabel(r'$N_{\mathrm{perm}}$  (beads)')
    axC.set_title('(a) cumulative solvent through support'); axC.grid(alpha=0.3)

    axR.plot(pstep, dN, '-', color=WONG['vermillion'], lw=1.0, alpha=0.30, label=r'$dN/dt$ (raw)')
    axR.plot(pstep, dN_roll, '-', color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label='rolling mean (21 pts)')
    axR.axhline(0, color='k', ls='--', lw=0.9, alpha=0.5)
    axR.set_xlabel('step'); axR.set_ylabel(r'$dN/dt$  (beads / step)')
    axR.set_title('(b) permeation rate'); axR.grid(alpha=0.3); axR.legend(fontsize=13, loc='best')
    # secondary axis: flux J = (dN/dt)/A, scale synced to the rolling rate axis
    axR2 = axR.twinx()
    axR2.set_ylabel(r'flux $J=(dN/dt)/A$  ($\sigma^{-2}\,\mathrm{step}^{-1}$)')
    axR2.set_ylim(np.array(axR.get_ylim()) / piston_area)

    out = PLOT_DIR / f'permeate_count_rate_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
    print(f'  N_perm: {Nperm[0]:.0f} -> {Nperm[-1]:.0f} beads over '
          f'{pstep[-1]-pstep[0]} steps')
else:
    print('skipped — permeate_count file missing')

## 6 — Solvent stress: partial vs. solvent-only (ss)

In compression the network stress was simply the total minus the (uniform)
reservoir pressure.  Under permeation the solvent pressure **drops from feed to
permeate**, so before decomposing anything we first look at the solvent stress
itself two ways, side by side:

- **left — solvent partial stress** $\sigma_{s,\alpha\alpha}$: the group-based
  virial of every force *on* a solvent atom (solvent–solvent **and**
  solvent–polymer / solvent–piston), from `sigma{zz,xx,yy}_solvent`.
- **right — solvent-only (ss) stress** $\sigma_{s,\alpha\alpha}^{ss}$: the
  solvent–solvent pair virial only, reconstructed from the `pairs` dump.

Rows are $\sigma_{zz}$, $\sigma_{xx}$, $\sigma_{yy}$ and the isotropic average
$\tfrac13(\sigma_{xx}+\sigma_{yy}+\sigma_{zz})$.  Each panel is a time evolution
(colour = timestep, bold black = final).

In [ ]:
# ==========================================================================
#  6 — SOLVENT PARTIAL STRESS  vs  SOLVENT-ONLY (ss) STRESS  (evolution)
#      rows: zz, xx, yy, isotropic average   |   cols: partial (L), ss-only (R)
# ==========================================================================
# partial (group-based) solvent stress, per frame, per component
part_zz = np.array(sig_s_zz); part_xx = np.array(sig_s_xx); part_yy = np.array(sig_s_yy)
part_iso = (part_xx + part_yy + part_zz) / 3.0

fig, axes = plt.subplots(4, 2, figsize=(17, 26), constrained_layout=True)
fig.suptitle(f'Solvent stress: partial (left) vs solvent-only ss (right):  {sim_name}',
             fontsize=14, fontweight='bold')

_rows = [
    ('zz',  part_zz,  ss_zz,  r'\sigma_{s,zz}'),
    ('xx',  part_xx,  ss_xx,  r'\sigma_{s,xx}'),
    ('yy',  part_yy,  ss_yy,  r'\sigma_{s,yy}'),
    ('iso', part_iso, ss_iso, r'\bar{\sigma}_{s}'),
]
# reference overlays keyed by row tag ('iso' uses the isotropic-average refs)
_ref_solv_row = {'zz': (ref_solv_ci or {}).get('zz'), 'xx': (ref_solv_ci or {}).get('xx'),
                 'yy': (ref_solv_ci or {}).get('yy'), 'iso': ref_solv_iso_ci}
_ref_ss_row   = {'zz': (ref_ss_ci or {}).get('zz'), 'xx': (ref_ss_ci or {}).get('xx'),
                 'yy': (ref_ss_ci or {}).get('yy'), 'iso': ref_ss_iso_ci}
for r, (tag, part_stack, ss_stack, sym) in enumerate(_rows):
    # left: partial solvent stress
    ts_e, ev = subsample(prod_ts, part_stack, n_curves)
    plot_evolution(axes[r, 0], z_coords, ts_e, ev,
                   rf'${sym}(z,t)$', rf'partial  ${sym}$')
    overlay_reference(axes[r, 0], z_coords, _ref_solv_row.get(tag))
    # right: solvent-only ss stress
    if ss_ts is not None and ss_stack is not None and len(ss_ts):
        ts_s, ev_s = subsample(ss_ts, ss_stack, n_curves)
        plot_evolution(axes[r, 1], z_coords, ts_s, ev_s,
                       rf'${sym}^{{ss}}(z,t)$', rf'solvent-only  ${sym}^{{ss}}$')
        overlay_reference(axes[r, 1], z_coords, _ref_ss_row.get(tag))
    else:
        axes[r, 1].text(0.5, 0.5, 'ss stress\nunavailable', ha='center', va='center',
                        transform=axes[r, 1].transAxes)
        axes[r, 1].set_title(rf'solvent-only  ${sym}^{{ss}}$')

out = PLOT_DIR / f'solvent_partial_vs_ss_stress_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()